# `sim.clock`: simulated time
**Host only.** The simulation never runs on the board, so this notebook has no hardware
section. Every cell runs in ordinary Python (CPython) on this computer.

# TODO:
Nothing open.

Decided (2026-11-16): simulated time moves only when code sleeps (`compat.sleep_ms`,
`sleep_us`) or calls `advance`. Nothing runs "in the background": a fake works out what
happened since it was last asked (measurement done, sample ready, a signal's new value) when
the driver next talks to it. That keeps the simulation single-threaded and repeatable.

# Python file
location: `src/micropython_i2c/sim/clock.py`

tests: `tests/test_sim_clock.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/sim/clock.py
"""``SimClock``: simulated time for drivers and fakes on the host.

Install it with ``with SimClock() as clock:``. Inside the block every ``compat`` timing call
(``sleep_ms``, ``sleep_us``, ``ticks_ms``, ``ticks_us``) uses simulated time: a sleep returns at
once and moves the clock forward by exactly the time asked for. Time moves only then, or when
the test calls ``advance``. A one-hour scenario runs in milliseconds and the same way every
time.

Host only: ``compat.clock_set`` refuses on MicroPython.

Generated by ``dev_notebooks/sim_clock_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None.

## Internal
`compat` for `clock_set`, which makes every timing function use this clock.

In [ ]:
%%extract ../src/micropython_i2c/sim/clock.py -a
from micropython_i2c.compat import clock_set

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_sim_clock.py
import time

import pytest

from micropython_i2c.compat import clock_get, sleep_ms, sleep_us, ticks_ms, ticks_us
from micropython_i2c.sim.clock import SimClock

Notebook-only: the imports the validation and test cells use.

In [ ]:
import time

import pytest

from micropython_i2c.compat import clock_get, sleep_ms, sleep_us, ticks_ms, ticks_us

# Intent
**Purpose:** run drivers and fakes without waiting, and the same way every run.

**Why it exists:** a driver waits real milliseconds (the SHTC3 12.1 ms per measurement, the
VL53L1X a whole timing budget), and a fake times its measurements on the clock. On the wall
clock a test of an hour-long scenario would take an hour, and timing races would make it
flaky. On `SimClock` the driver's `sleep_ms(12)` moves time forward 12 ms at once, and the
fake sees exactly 12 ms pass.

```python
with SimClock() as clock:
    sht.measure()            # its 12.1 ms wait takes no real time
    clock.advance(60)        # one simulated minute later
    clock.now_s()            # 60.0121
```

**Not its job:** threads. Simulated time is for single-threaded code; two threads sleeping on
one simulated clock would each move it forward.

## What the module defines

| Name | What it is |
|---|---|
| `SimClock` | simulated time; a context manager that installs itself as the `compat` clock |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/sim/clock.py -a
__all__ = ["SimClock"]

## `SimClock`
Time is a whole number of microseconds, so repeated small sleeps add up exactly (no float
drift). `install` / `uninstall` swap it in and out of `compat`; the `with` form does both and
restores whatever clock was there before, so blocks nest.

In [ ]:
%%extract ../src/micropython_i2c/sim/clock.py -a
class SimClock:
    """Simulated time. Moves only on a sleep or ``advance``.

    Args:
        start_s: Simulated time at creation, in seconds.

    Attributes:
        now_us: Simulated time in microseconds.
        sleeps: How many sleeps the clock has served (handy to check a driver waited).
    """

    def __init__(self, start_s=0):
        self.now_us = int(round(start_s * 1_000_000))
        self.sleeps = 0
        self._previous = []

    # The clock interface compat calls.

    def ticks_us(self):
        """Simulated microseconds."""
        return self.now_us

    def sleep_us(self, us):
        """Move time forward ``us`` microseconds, at once. A negative sleep is a no-op."""
        self.sleeps += 1
        if us > 0:
            self.now_us += int(us)

    # For tests and scenarios.

    def advance(self, seconds):
        """Move time forward ``seconds`` (a float), as if the program had slept that long.

        Raises:
            ValueError: If ``seconds`` is negative: simulated time never goes back.
        """
        if seconds < 0:
            raise ValueError("cannot advance by {} s: time never goes back".format(seconds))
        self.now_us += int(round(seconds * 1_000_000))

    def now_s(self):
        """Simulated time in seconds (a float)."""
        return self.now_us / 1_000_000

    def install(self):
        """Make this the ``compat`` clock. ``uninstall`` puts the previous one back."""
        self._previous.append(clock_set(self))
        return self

    def uninstall(self):
        """Restore the clock that was in place before the matching ``install``.

        Raises:
            RuntimeError: If this clock is not installed.
        """
        if not self._previous:
            raise RuntimeError("this SimClock is not installed")
        clock_set(self._previous.pop())

    def __enter__(self):
        return self.install()

    def __exit__(self, *exc):
        self.uninstall()
        return False

### validation/test

#### validation: nominal usage
A 10-minute sleep inside the block takes no real time, and both counters move by exactly 10
minutes. After the block, the wall clock is back.

In [ ]:
wall = time.monotonic()
with SimClock() as clock:
    sleep_ms(600_000)
    sleep_us(250)
    print("simulated: {} s | ticks_ms: {} | sleeps served: {}".format(clock.now_s(), ticks_ms(), clock.sleeps))
print("real time spent: {:.4f} s | wall clock back:".format(time.monotonic() - wall), type(clock_get()).__name__)

`advance` moves time without a sleep (a test standing in for "the program did something else
for a minute"), and blocks nest: the inner clock is used inside, the outer one after.

In [ ]:
outer = SimClock(start_s=100)
with outer:
    outer.advance(60)
    with SimClock() as inner:
        sleep_ms(5)
        print("inner:", ticks_ms(), "ms")
    print("outer again:", ticks_ms(), "ms | inner left at:", inner.now_s(), "s")

#### validation: anti-nominal usage

In [ ]:
c = SimClock()
try:
    c.advance(-1)  # simulated time never goes back
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("advance(-1) should have raised ValueError")
finally:
    print("time unchanged:", c.now_us == 0)

In [ ]:
c = SimClock()
try:
    c.uninstall()  # never installed
except RuntimeError as err:
    print("raised RuntimeError, as expected:\n    {}".format(err))
else:
    raise AssertionError("uninstall() of a clock never installed should have raised RuntimeError")
finally:
    print("compat clock untouched:", type(clock_get()).__name__)

#### unit

In [ ]:
%%extract ../tests/test_sim_clock.py -a --strip-ipytest
%%ipytest


def test_sleeps_take_no_real_time_and_add_up_exactly():
    wall = time.monotonic()
    with SimClock() as clock:
        for _ in range(1000):
            sleep_us(1)
        sleep_ms(60_000)
    assert clock.now_us == 60_001_000
    assert time.monotonic() - wall < 1.0

In [ ]:
%%extract ../tests/test_sim_clock.py -a --strip-ipytest
%%ipytest


def test_ticks_follow_the_simulated_time():
    with SimClock(start_s=2.5) as clock:
        assert ticks_ms() == 2500
        clock.advance(0.0015)
        assert ticks_us() == 2_501_500

In [ ]:
%%extract ../tests/test_sim_clock.py -a --strip-ipytest
%%ipytest


def test_the_previous_clock_comes_back_and_blocks_nest():
    before = clock_get()
    outer, inner = SimClock(), SimClock()
    with outer:
        with inner:
            assert clock_get() is inner
        assert clock_get() is outer
    assert clock_get() is before

In [ ]:
%%extract ../tests/test_sim_clock.py -a --strip-ipytest
%%ipytest


def test_the_clock_is_restored_after_an_error():
    before = clock_get()
    with pytest.raises(KeyError):
        with SimClock():
            raise KeyError("boom")
    assert clock_get() is before

In [ ]:
%%extract ../tests/test_sim_clock.py -a --strip-ipytest
%%ipytest


def test_time_never_goes_back():
    c = SimClock()
    with pytest.raises(ValueError, match="never goes back"):
        c.advance(-0.001)
    c.sleep_us(-5)
    assert c.now_us == 0

In [ ]:
%%extract ../tests/test_sim_clock.py -a --strip-ipytest
%%ipytest


def test_uninstall_without_install_is_refused():
    with pytest.raises(RuntimeError, match="not installed"):
        SimClock().uninstall()